In [1]:
from pathlib import Path
import json
import time
import uuid
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("..").resolve()

CACHE_DIR = PROJECT_ROOT / "notebook_cache"

INGESTION_DIR = CACHE_DIR / "01_ingestion"
SEMANTIC_DIR = CACHE_DIR / "02_semantic_retrieval"
GRAPH_DIR = CACHE_DIR / "03_graph_build"
EVAL_DIR = CACHE_DIR / "04_evaluation"

EXECUTOR_CACHE = CACHE_DIR / "05_graphrag_executor"
EXECUTOR_CACHE.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Executor cache:", EXECUTOR_CACHE)

Project root: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent
Executor cache: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor


In [2]:
required_files = {
    "chunks": INGESTION_DIR / "all_chunks.json",
    "metadata": INGESTION_DIR / "metadata.json",
    "queries": INGESTION_DIR / "labeled_query_set.json",
    "chunk_stats": INGESTION_DIR / "chunk_stats.json",
    "embeddings": SEMANTIC_DIR / "embeddings.npy",
    "topic_assignments": GRAPH_DIR / "neo4j_topic_assignments.csv",
    "graph_summary": GRAPH_DIR / "neo4j_graph_summary.csv",
}

verification_df = pd.DataFrame([
    {
        "name": name,
        "path": str(path),
        "exists": path.exists()
    }
    for name, path in required_files.items()
])

display(verification_df)

missing = verification_df[verification_df["exists"] == False]

if len(missing) > 0:
    raise FileNotFoundError("Some required previous outputs are missing. Run earlier notebooks first.")
else:
    print("All required previous outputs found.")

,name,path,exists
0,chunks,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\01_ingestion\all_chunks.json,True
1,metadata,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\01_ingestion\metadata.json,True
2,queries,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\01_ingestion\labeled_query_set.json,True
3,chunk_stats,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\01_ingestion\chunk_stats.json,True
4,embeddings,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\02_semantic_retrieval\embeddings.npy,True
5,topic_assignments,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\03_graph_build\neo4j_topic_assignments.csv,True
6,graph_summary,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\03_graph_build\neo4j_graph_summary.csv,True


All required previous outputs found.


In [3]:
with open(required_files["chunks"], "r", encoding="utf-8") as f:
    chunks = json.load(f)

with open(required_files["metadata"], "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(required_files["queries"], "r", encoding="utf-8") as f:
    queries = json.load(f)

with open(required_files["chunk_stats"], "r", encoding="utf-8") as f:
    chunk_stats = json.load(f)

embeddings = np.load(required_files["embeddings"])

topic_df = pd.read_csv(required_files["topic_assignments"])
graph_summary_df = pd.read_csv(required_files["graph_summary"])

print("Chunks loaded:", len(chunks))
print("Metadata records:", len(metadata))
print("Queries loaded:", len(queries))
print("Embeddings shape:", embeddings.shape)
print("Topic assignments:", len(topic_df))

Chunks loaded: 6728
Metadata records: 208
Queries loaded: 20
Embeddings shape: (6728, 384)
Topic assignments: 776


In [4]:
chunks_df = pd.DataFrame(chunks)
metadata_df = pd.DataFrame(metadata)
chunk_stats_df = pd.DataFrame(chunk_stats)

print("Chunks sample")
display(chunks_df[["chunk_id", "paper_id", "page_start", "page_end", "word_count"]].head())

print("Metadata sample")
display(metadata_df[["paper_id", "title", "authors", "year", "page_count"]].head())

print("Chunk statistics")
display(chunk_stats_df.describe())

print("Topic assignment sample")
display(topic_df.head())

print("Graph summary")
display(graph_summary_df.head())

Chunks sample


,chunk_id,paper_id,page_start,page_end,word_count
0,paper1_chunk_0000,paper1,1,1,400
1,paper1_chunk_0001,paper1,1,2,400
2,paper1_chunk_0002,paper1,2,3,400
3,paper1_chunk_0003,paper1,2,3,400
4,paper1_chunk_0004,paper1,3,3,400


Metadata sample


,paper_id,title,authors,year,page_count
0,paper1,FlexSQL: Flexible Exploration and Execution Make Better Text-to-SQL Agents,Quang Hieu Pham; Yang He; Ping Nie; Canwen Xu; Davood Rafiei; Yuepeng Wang; Xi Ye; Jocelyn Qiaochu Chen,2026,18
1,paper2,Reinforcement Learning for LLM-based Multi-Agent Systems through Orchestration Traces,Chenchen Zhang,2026,71
2,paper3,mdok-style at SemEval-2026 Task 10: Finetuning LLMs for Conspiracy Detection,Dominik Macko,2026,5
3,paper4,mdok-style at SemEval-2026 Task 9: Finetuning LLMs for Multilingual Polarization Detection,Dominik Macko; Alok Debnath; Jakub Simko,2026,8
4,paper5,Fuzzy Fingerprinting Encoder Pre-trained Language Models for Emotion Recognition in Conversations: Human Assessment ...,Patrícia Pereira; Helena Moniz; Joao Paulo Carvalho,2026,12


Chunk statistics


,num_chunks,total_words_raw,overlap_words,unique_word_count,chunks_word_sum
count,208.000000,208.000000,208.000000,208.000000,208.000000
mean,32.346154,11149.807692,1567.307692,11149.807692,12713.750000
std,25.009281,8751.770546,1250.464054,8751.770546,10001.887553
min,7.000000,2361.000000,300.000000,2361.000000,2661.000000
25%,20.000000,6957.500000,950.000000,6957.500000,7907.500000
50%,27.000000,9152.000000,1300.000000,9152.000000,10429.500000
75%,38.000000,12990.750000,1850.000000,12990.750000,14819.750000
max,231.000000,80720.000000,11500.000000,80720.000000,92220.000000


Topic assignment sample


,paper_id,title,topic,topic_type,source,evidence,confidence,tfidf_score,doc_frequency
0,paper1,FlexSQL: Flexible Exploration and Execution Make Better Text-to-SQL Agents,Text-to-SQL,controlled,controlled_title,text-to-sql,0.95,NaN,NaN
1,paper1,FlexSQL: Flexible Exploration and Execution Make Better Text-to-SQL Agents,Code Generation,controlled,controlled_abstract,coding agent,0.82,NaN,NaN
2,paper1,FlexSQL: Flexible Exploration and Execution Make Better Text-to-SQL Agents,Reasoning,controlled,controlled_abstract,reasoning,0.82,NaN,NaN
3,paper2,Reinforcement Learning for LLM-based Multi-Agent Systems through Orchestration Traces,Large Language Models,controlled,controlled_title,llm,0.95,NaN,NaN
4,paper2,Reinforcement Learning for LLM-based Multi-Agent Systems through Orchestration Traces,LLM Agents,controlled,controlled_abstract,llm agents,0.82,NaN,NaN


Graph summary


,graph_item,count
0,papers,208
1,authors,951
2,topics,286
3,WROTE relationships,981
4,HAS_TOPIC relationships,776


In [5]:
metadata_lookup = {
    m["paper_id"]: m
    for m in metadata
    if "paper_id" in m
}

chunks_by_id = {
    c["chunk_id"]: c
    for c in chunks
    if "chunk_id" in c
}

chunks_by_paper = {}

for c in chunks:
    pid = c.get("paper_id")
    if pid:
        chunks_by_paper.setdefault(pid, []).append(c)

print("Unique papers in chunks:", len(chunks_by_paper))
print("Metadata lookup:", len(metadata_lookup))
print("Chunk lookup:", len(chunks_by_id))

Unique papers in chunks: 208
Metadata lookup: 208
Chunk lookup: 6728


In [6]:
chunk_index_by_id = {
    c["chunk_id"]: i
    for i, c in enumerate(chunks)
    if "chunk_id" in c
}

chunk_indices_by_paper = {}

for i, c in enumerate(chunks):
    pid = c.get("paper_id")
    if pid:
        chunk_indices_by_paper.setdefault(pid, []).append(i)


def normalize_values(values):
    values = np.array(values, dtype=float)

    if len(values) == 0:
        return values

    min_v = values.min()
    max_v = values.max()

    if max_v == min_v:
        return np.ones_like(values) if max_v > 0 else np.zeros_like(values)

    return (values - min_v) / (max_v - min_v)


print("Chunk index ready:", len(chunk_index_by_id))
print("Papers with chunks:", len(chunk_indices_by_paper))

Chunk index ready: 6728
Papers with chunks: 208


In [7]:
texts = [c.get("text", "") for c in chunks]

tfidf_vectorizer = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

print("TF-IDF matrix:", tfidf_matrix.shape)
print("Embeddings:", embeddings.shape)

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    dense_available = True
    print("Dense query model loaded.")
except Exception as e:
    embedding_model = None
    dense_available = False
    print("Dense model not available. Dense search will be skipped.")
    print(e)

TF-IDF matrix: (6728, 50000)
Embeddings: (6728, 384)


d:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 41882.05it/s]


Dense query model loaded.


In [8]:
def add_metadata_to_chunk(chunk):
    pid = chunk.get("paper_id")
    meta = metadata_lookup.get(pid, {})

    merged = dict(chunk)
    merged.setdefault("title", meta.get("title", "unknown"))
    merged.setdefault("authors", meta.get("authors", "unknown"))
    merged.setdefault("year", meta.get("year", "unknown"))
    merged.setdefault("filename", meta.get("filename", "unknown"))

    return merged


def make_retrieval_result(rank, score, chunk):
    chunk = add_metadata_to_chunk(chunk)

    text = chunk.get("text", "")

    return {
        "rank": rank,
        "score": float(score),
        "chunk_id": chunk.get("chunk_id"),
        "paper_id": chunk.get("paper_id"),
        "title": chunk.get("title", "unknown"),
        "authors": chunk.get("authors", "unknown"),
        "year": chunk.get("year", "unknown"),
        "page_start": chunk.get("page_start", "unknown"),
        "page_end": chunk.get("page_end", "unknown"),
        "snippet": text[:300] + "..." if len(text) > 300 else text,
    }


def tfidf_search(query, top_k=5):
    q_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [
        make_retrieval_result(i + 1, scores[idx], chunks[idx])
        for i, idx in enumerate(top_idx)
    ]


def dense_search_cached(query, top_k=5):
    if not dense_available:
        return []

    q_emb = embedding_model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, embeddings).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [
        make_retrieval_result(i + 1, scores[idx], chunks[idx])
        for i, idx in enumerate(top_idx)
    ]


def hybrid_search_cached(query, top_k=5, alpha=0.5, candidate_k=30):
    tfidf_hits = tfidf_search(query, top_k=candidate_k)
    dense_hits = dense_search_cached(query, top_k=candidate_k)

    score_map = {}

    for hit in tfidf_hits:
        cid = hit["chunk_id"]
        score_map.setdefault(cid, {"chunk_id": cid, "tfidf": 0.0, "dense": 0.0})
        score_map[cid]["tfidf"] = hit["score"]

    for hit in dense_hits:
        cid = hit["chunk_id"]
        score_map.setdefault(cid, {"chunk_id": cid, "tfidf": 0.0, "dense": 0.0})
        score_map[cid]["dense"] = hit["score"]

    rows = []

    for cid, scores in score_map.items():
        chunk = chunks_by_id[cid]
        final_score = alpha * scores["dense"] + (1 - alpha) * scores["tfidf"]
        rows.append((final_score, chunk))

    rows = sorted(rows, key=lambda x: x[0], reverse=True)[:top_k]

    return [
        make_retrieval_result(i + 1, score, chunk)
        for i, (score, chunk) in enumerate(rows)
    ]

In [9]:
def get_seed_paper_ids(seed_chunks, max_papers=3):
    seen = []

    for hit in seed_chunks:
        pid = hit.get("paper_id")

        if pid and pid not in seen:
            seen.append(pid)

        if len(seen) >= max_papers:
            break

    return seen


def expand_by_shared_topics(seed_paper_ids, limit=5):
    if len(seed_paper_ids) == 0 or limit <= 0:
        return []

    if "paper_id" not in topic_df.columns:
        raise ValueError("topic_df must contain paper_id column.")

    possible_topic_cols = [c for c in topic_df.columns if "topic" in c.lower()]
    if not possible_topic_cols:
        raise ValueError("topic_df must contain a topic column.")

    topic_col = possible_topic_cols[0]

    seed_topics = topic_df[
        topic_df["paper_id"].isin(seed_paper_ids)
    ][topic_col].dropna().unique().tolist()

    related = topic_df[
        topic_df[topic_col].isin(seed_topics)
        & ~topic_df["paper_id"].isin(seed_paper_ids)
    ]

    expanded = []

    for pid, group in related.groupby("paper_id"):
        topics = group[topic_col].dropna().unique().tolist()

        expanded.append({
            "paper_id": pid,
            "graph_score": float(len(topics)),
            "reasons": [f"shared_topic: {t}" for t in topics]
        })

    expanded = sorted(
        expanded,
        key=lambda x: x["graph_score"],
        reverse=True
    )

    return expanded[:limit]

In [10]:
def make_evidence_item(
    chunk,
    source,
    retrieval_score=0.0,
    graph_score=0.0,
    graph_reasons=None
):
    graph_reasons = graph_reasons or []

    chunk = add_metadata_to_chunk(chunk)

    has_pages = (
        chunk.get("page_start") not in [None, "unknown"]
        and chunk.get("page_end") not in [None, "unknown"]
    )

    quality_score = 1.0 if has_pages else 0.5

    return {
        "rank": None,
        "chunk_id": chunk.get("chunk_id"),
        "paper_id": chunk.get("paper_id"),
        "title": chunk.get("title", "unknown"),
        "authors": chunk.get("authors", "unknown"),
        "year": chunk.get("year", "unknown"),
        "page_start": chunk.get("page_start", "unknown"),
        "page_end": chunk.get("page_end", "unknown"),
        "source": source,
        "retrieval_score": float(retrieval_score),
        "graph_score": float(graph_score),
        "quality_score": quality_score,
        "final_score": 0.0,
        "graph_reasons": graph_reasons,
        "snippet": chunk.get("text", "")[:300],
    }

In [11]:
def score_chunks_within_paper(question, paper_id, top_n=2, alpha=0.5):
    """
    Rank chunks inside one graph-expanded paper by relevance to the question.
    This prevents GraphRAG from blindly taking the first chunks of the paper.
    """

    indices = chunk_indices_by_paper.get(paper_id, [])

    if not indices:
        return []

    # TF-IDF relevance
    q_tfidf = tfidf_vectorizer.transform([question])
    paper_tfidf_matrix = tfidf_matrix[indices]
    tfidf_scores = cosine_similarity(q_tfidf, paper_tfidf_matrix).flatten()

    # Dense relevance
    if dense_available:
        q_emb = embedding_model.encode([question], convert_to_numpy=True)
        paper_embeddings = embeddings[indices]
        dense_scores = cosine_similarity(q_emb, paper_embeddings).flatten()
    else:
        dense_scores = np.zeros_like(tfidf_scores)

    tfidf_norm = normalize_values(tfidf_scores)
    dense_norm = normalize_values(dense_scores)

    local_scores = alpha * dense_norm + (1 - alpha) * tfidf_norm

    ranked_local = np.argsort(local_scores)[::-1][:top_n]

    results = []

    for local_idx in ranked_local:
        global_idx = indices[local_idx]
        chunk = chunks[global_idx]

        raw_relevance_score = (
            alpha * dense_scores[local_idx]
            + (1 - alpha) * tfidf_scores[local_idx]
        )

        results.append({
            "chunk": chunk,
            "question_relevance_score_raw": float(raw_relevance_score),
            "question_relevance_score_local": float(local_scores[local_idx]),
            "tfidf_score_within_graph_paper": float(tfidf_scores[local_idx]),
            "dense_score_within_graph_paper": float(dense_scores[local_idx]),
        })

    return results


def collect_graph_chunks_relevant(
    question,
    graph_papers,
    max_chunks_per_graph_paper=2,
    alpha=0.5
):
    """
    Graph selects related papers.
    Retrieval selects the most question-relevant chunks from those papers.
    """

    evidence = []

    max_graph_score = max(
        [gp.get("graph_score", 0.0) for gp in graph_papers],
        default=1.0
    )

    for gp in graph_papers:
        best_chunks = score_chunks_within_paper(
            question=question,
            paper_id=gp["paper_id"],
            top_n=max_chunks_per_graph_paper,
            alpha=alpha
        )

        graph_score_raw = gp.get("graph_score", 0.0)
        graph_score_norm = graph_score_raw / max_graph_score if max_graph_score else 0.0

        for row in best_chunks:
            item = make_evidence_item(
                chunk=row["chunk"],
                source="graph",
                retrieval_score=row["question_relevance_score_raw"],
                graph_score=graph_score_norm,
                graph_reasons=gp.get("reasons", []),
            )

            item["graph_score_raw"] = graph_score_raw
            item["question_relevance_score_local"] = row["question_relevance_score_local"]
            item["tfidf_score_within_graph_paper"] = row["tfidf_score_within_graph_paper"]
            item["dense_score_within_graph_paper"] = row["dense_score_within_graph_paper"]

            evidence.append(item)

    return evidence

In [12]:
def merge_evidence(vector_evidence, graph_evidence):
    merged = {}

    for item in vector_evidence + graph_evidence:
        cid = item["chunk_id"]

        if cid not in merged:
            merged[cid] = item
        else:
            existing = merged[cid]
            existing["source"] = "both"
            existing["retrieval_score"] = max(existing["retrieval_score"], item["retrieval_score"])
            existing["graph_score"] = max(existing["graph_score"], item["graph_score"])
            existing["graph_reasons"] = list(set(existing["graph_reasons"] + item["graph_reasons"]))

    return list(merged.values())


def rerank_evidence(
    evidence,
    retrieval_weight,
    graph_weight,
    quality_weight,
    max_total_evidence_chunks
):
    """
    Normalize retrieval and graph scores across all candidate evidence before blending.
    This prevents raw graph scores from dominating vector scores.
    """

    if len(evidence) == 0:
        return []

    retrieval_raw = [item.get("retrieval_score", 0.0) for item in evidence]
    graph_raw = [item.get("graph_score", 0.0) for item in evidence]

    retrieval_norm = normalize_values(retrieval_raw)
    graph_norm = normalize_values(graph_raw)

    for i, item in enumerate(evidence):
        item["retrieval_score_raw"] = float(retrieval_raw[i])
        item["graph_score_raw_for_rerank"] = float(graph_raw[i])

        item["retrieval_score"] = float(retrieval_norm[i])
        item["graph_score"] = float(graph_norm[i])

        item["final_score"] = round(
            retrieval_weight * item["retrieval_score"]
            + graph_weight * item["graph_score"]
            + quality_weight * item["quality_score"],
            4
        )

    ranked = sorted(evidence, key=lambda x: x["final_score"], reverse=True)
    final = ranked[:max_total_evidence_chunks]

    for i, item in enumerate(final, start=1):
        item["rank"] = i

    return final


def build_citations(final_evidence):
    citations = []

    for item in final_evidence:
        page_start = item.get("page_start", "unknown")
        page_end = item.get("page_end", "unknown")

        pages = str(page_start) if page_start == page_end else f"{page_start}-{page_end}"

        citations.append({
            "rank": item["rank"],
            "paper_id": item["paper_id"],
            "title": item["title"],
            "chunk_id": item["chunk_id"],
            "pages": pages,
            "source": item["source"],
            "graph_reasons": item["graph_reasons"],
        })

    return citations

In [13]:
MODE_PRESETS = {
    "vector_only": {
        "use_graph": False,
        "retrieval_weight": 1.0,
        "graph_weight": 0.0,
        "quality_weight": 0.0,
    },

    "graph_guided": {
        "use_graph": True,
        "retrieval_weight": 0.5,
        "graph_weight": 0.4,
        "quality_weight": 0.1,
    },

    "hybrid_graphrag": {
        "use_graph": True,
        "retrieval_weight": 0.8,
        "graph_weight": 0.1,
        "quality_weight": 0.1,
    },
}

In [14]:
def generate_simple_answer(question, final_evidence):
    if len(final_evidence) == 0:
        return "No sufficient evidence was found."

    lines = []

    for item in final_evidence[:4]:
        pages = (
            str(item["page_start"])
            if item["page_start"] == item["page_end"]
            else f"{item['page_start']}-{item['page_end']}"
        )

        lines.append(
            f"- {item['title']} pages {pages}: {item['snippet']}"
        )

    return (
        f"Question: {question}\n\n"
        "Based on the selected evidence, the answer is supported by these papers:\n\n"
        + "\n".join(lines)
    )


def run_graphrag_executor(
    question,
    mode="hybrid_graphrag",
    retrieval_method="hybrid",
    alpha=0.5,
    top_k=5,
    candidate_k=30,
    seed_papers=3,
    expand_k=5,
    max_chunks_per_graph_paper=2,
    max_total_evidence_chunks=8,
    retrieval_weight=None,
    graph_weight=None,
    quality_weight=None,
    save_trace=True
):
    run_id = str(uuid.uuid4())
    t_total = time.time()

    cfg = MODE_PRESETS[mode].copy()

    if retrieval_weight is not None:
        cfg["retrieval_weight"] = retrieval_weight
    if graph_weight is not None:
        cfg["graph_weight"] = graph_weight
    if quality_weight is not None:
        cfg["quality_weight"] = quality_weight

    timing = {}

    t0 = time.time()

    if retrieval_method == "tfidf":
        seed_chunks = tfidf_search(question, top_k=top_k)
    elif retrieval_method == "dense":
        seed_chunks = dense_search_cached(question, top_k=top_k)
    else:
        seed_chunks = hybrid_search_cached(
            question,
            top_k=top_k,
            alpha=alpha,
            candidate_k=candidate_k
        )

    timing["retrieval_ms"] = round((time.time() - t0) * 1000, 2)

    seed_paper_ids = get_seed_paper_ids(seed_chunks, max_papers=seed_papers)

    vector_evidence = [
        make_evidence_item(
            chunk=chunks_by_id[hit["chunk_id"]],
            source="vector",
            retrieval_score=hit["score"],
            graph_score=0.0,
            graph_reasons=[]
        )
        for hit in seed_chunks
    ]

    t0 = time.time()

    if cfg["use_graph"]:
        graph_papers = expand_by_shared_topics(seed_paper_ids, limit=expand_k)

        graph_evidence = collect_graph_chunks_relevant(
            question=question,
            graph_papers=graph_papers,
            max_chunks_per_graph_paper=max_chunks_per_graph_paper,
            alpha=alpha
        )
    else:
        graph_papers = []
        graph_evidence = []

    timing["graph_ms"] = round((time.time() - t0) * 1000, 2)

    candidate_evidence = merge_evidence(vector_evidence, graph_evidence)

    t0 = time.time()

    final_evidence = rerank_evidence(
        evidence=candidate_evidence,
        retrieval_weight=cfg["retrieval_weight"],
        graph_weight=cfg["graph_weight"],
        quality_weight=cfg["quality_weight"],
        max_total_evidence_chunks=max_total_evidence_chunks
    )

    timing["rerank_ms"] = round((time.time() - t0) * 1000, 2)

    t0 = time.time()
    answer = generate_simple_answer(question, final_evidence)
    timing["generation_ms"] = round((time.time() - t0) * 1000, 2)
    timing["total_ms"] = round((time.time() - t_total) * 1000, 2)

    citations = build_citations(final_evidence)

    trace = {
        "run_id": run_id,
        "mode": mode,
        "question": question,

        "parameters": {
            "retrieval_method": retrieval_method,
            "alpha": alpha,
            "top_k": top_k,
            "candidate_k": candidate_k,
            "seed_papers": seed_papers,
            "expand_k": expand_k,
            "max_chunks_per_graph_paper": max_chunks_per_graph_paper,
            "max_total_evidence_chunks": max_total_evidence_chunks,
            "retrieval_weight": cfg["retrieval_weight"],
            "graph_weight": cfg["graph_weight"],
            "quality_weight": cfg["quality_weight"],
            "use_graph": cfg["use_graph"],
        },

        "retrieval_trace": {
            "seed_chunks": seed_chunks,
            "seed_paper_ids": seed_paper_ids,
        },

        "graph_trace": {
            "enabled": cfg["use_graph"],
            "expanded_papers": graph_papers,
            "path_type": "shared_topic_from_cached_neo4j_topic_assignments",
        },

        "candidate_evidence": candidate_evidence,
        "final_evidence": final_evidence,
        "answer": answer,
        "citations": citations,
        "timing": timing,

        "quality_checks": {
            "num_seed_chunks": len(seed_chunks),
            "num_seed_papers": len(seed_paper_ids),
            "num_graph_papers": len(graph_papers),
            "num_candidate_chunks": len(candidate_evidence),
            "num_final_chunks": len(final_evidence),
            "citation_coverage": round(len(citations) / max(len(final_evidence), 1), 4),
            "has_page_ranges": all(c["pages"] != "unknown" for c in citations),
        }
    }

    if save_trace:
        out_path = EXECUTOR_CACHE / f"{run_id}_{mode}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(trace, f, indent=2, ensure_ascii=False)

        trace["saved_to"] = str(out_path)

    return trace

In [15]:
question = "How do recent papers evaluate retrieval augmented generation?"

trace = run_graphrag_executor(
    question=question,
    mode="hybrid_graphrag",
    retrieval_method="hybrid",
    top_k=5,
    seed_papers=3,
    expand_k=5,
    max_total_evidence_chunks=8
)

print(trace["answer"])
print("\nSaved trace:", trace.get("saved_to"))

Question: How do recent papers evaluate retrieval augmented generation?

Based on the selected evidence, the answer is supported by these papers:

- Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation pages 12: arXiv preprint arXiv:2310.13548 (2023). [63] Marco Siino, Mariana Falco, Daniele Croce, and Paolo Rosso. 2025. Exploring llms applications in law: A literature review on current legal nlp approaches. IEEE Access (2025). [64] Weihang Su, Yichen Tang, Qingyao Ai, Junxi Yan, Changyue Wang, Hongning Wan
- Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning pages 10: Kristina Toutanova. 2019. BERT: Pre-training of deep bidirectional transformers for language under- standing. In Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Tech- nologies, Volume 1 (Long and Short Papers), pages 4
- H-RAG at SemEval-2026 Task 8: Hierarchical

In [16]:
final_evidence_df = pd.DataFrame(trace["final_evidence"])

display(
    final_evidence_df[
        [
            "rank",
            "source",
            "paper_id",
            "title",
            "page_start",
            "page_end",
            "retrieval_score",
            "graph_score",
            "quality_score",
            "final_score",
            "graph_reasons"
        ]
    ]
)

,rank,source,paper_id,title,page_start,page_end,retrieval_score,graph_score,quality_score,final_score,graph_reasons
0,1,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,12,12,1.000000,0.0,1.0,0.9000,[]
1,2,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,10,10,0.823266,1.0,1.0,0.8586,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."
2,3,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,6,7,0.926649,0.0,1.0,0.8413,[]
3,4,graph,paper63,FT-RAG: A Fine-grained Retrieval-Augmented Generation Framework for Complex Table Reasoning,8,9,0.777740,1.0,1.0,0.8222,"[shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_topic: Information Retrie..."
4,5,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,7,7,0.894741,0.0,1.0,0.8158,[]
5,6,vector,paper12,Generation: A Controlled Empirical Study1,1,2,0.891289,0.0,1.0,0.8130,[]
6,7,vector,paper12,Generation: A Controlled Empirical Study1,14,14,0.877060,0.0,1.0,0.8016,[]
7,8,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,1,1,0.739531,1.0,1.0,0.7916,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."


In [17]:
citations_df = pd.DataFrame(trace["citations"])
display(citations_df)

,rank,paper_id,title,chunk_id,pages,source,graph_reasons
0,1,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,paper85_chunk_0027,12,vector,[]
1,2,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,paper70_chunk_0018,10,graph,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."
2,3,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0009,6-7,vector,[]
3,4,paper63,FT-RAG: A Fine-grained Retrieval-Augmented Generation Framework for Complex Table Reasoning,paper63_chunk_0017,8-9,graph,"[shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_topic: Information Retrie..."
4,5,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0010,7,vector,[]
5,6,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0000,1-2,vector,[]
6,7,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0017,14,vector,[]
7,8,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,paper70_chunk_0000,1,graph,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."


In [18]:
question = "How do recent papers evaluate retrieval augmented generation?"

mode_runs = []

for mode in ["vector_only", "graph_guided", "hybrid_graphrag"]:
    run = run_graphrag_executor(
        question=question,
        mode=mode,
        retrieval_method="hybrid",
        top_k=5,
        seed_papers=3,
        expand_k=5,
        max_chunks_per_graph_paper=2,
        max_total_evidence_chunks=8,
        save_trace=True
    )

    mode_runs.append({
        "mode": mode,
        "num_seed_chunks": run["quality_checks"]["num_seed_chunks"],
        "num_graph_papers": run["quality_checks"]["num_graph_papers"],
        "num_candidate_chunks": run["quality_checks"]["num_candidate_chunks"],
        "num_final_chunks": run["quality_checks"]["num_final_chunks"],
        "citation_coverage": run["quality_checks"]["citation_coverage"],
        "has_page_ranges": run["quality_checks"]["has_page_ranges"],
        "retrieval_ms": run["timing"]["retrieval_ms"],
        "graph_ms": run["timing"]["graph_ms"],
        "rerank_ms": run["timing"]["rerank_ms"],
        "generation_ms": run["timing"]["generation_ms"],
        "total_ms": run["timing"]["total_ms"],
        "saved_to": run.get("saved_to")
    })

comparison_df = pd.DataFrame(mode_runs)
display(comparison_df)

,mode,num_seed_chunks,num_graph_papers,num_candidate_chunks,num_final_chunks,citation_coverage,has_page_ranges,retrieval_ms,graph_ms,rerank_ms,generation_ms,total_ms,saved_to
0,vector_only,5,0,5,5,1.0,True,170.99,0.00,0.0,0.0,170.99,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\37cd45e7-193a-4b95-8418-c4576d...
1,graph_guided,5,5,15,8,1.0,True,121.83,98.03,0.0,0.0,219.86,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\aee7b740-d7f5-4d73-80dd-147ead...
2,hybrid_graphrag,5,5,15,8,1.0,True,128.52,93.34,0.0,0.0,221.87,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\13150942-3655-41ca-889c-19c710...


In [19]:
comparison_path = EXECUTOR_CACHE / "mode_comparison_sample.csv"
comparison_df.to_csv(comparison_path, index=False)

print("Saved comparison table:", comparison_path)
display(comparison_df)

Saved comparison table: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\mode_comparison_sample.csv


,mode,num_seed_chunks,num_graph_papers,num_candidate_chunks,num_final_chunks,citation_coverage,has_page_ranges,retrieval_ms,graph_ms,rerank_ms,generation_ms,total_ms,saved_to
0,vector_only,5,0,5,5,1.0,True,170.99,0.00,0.0,0.0,170.99,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\37cd45e7-193a-4b95-8418-c4576d...
1,graph_guided,5,5,15,8,1.0,True,121.83,98.03,0.0,0.0,219.86,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\aee7b740-d7f5-4d73-80dd-147ead...
2,hybrid_graphrag,5,5,15,8,1.0,True,128.52,93.34,0.0,0.0,221.87,D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\13150942-3655-41ca-889c-19c710...


In [20]:
def ask(
    question,
    mode="hybrid_graphrag",
    top_k=5,
    seed_papers=3,
    expand_k=5,
    max_chunks_per_graph_paper=2,
    max_total_evidence_chunks=8,
    retrieval_weight=None,
    graph_weight=None,
    quality_weight=None,
    return_trace=True,
    include_candidate_evidence=False,
    save_trace=True
):
    """
    Notebook-level API-ready /ask function.

    This is not FastAPI yet.
    It prepares the same response schema that the real /ask endpoint will return later.
    """

    trace = run_graphrag_executor(
        question=question,
        mode=mode,
        retrieval_method="hybrid",
        alpha=0.5,
        top_k=top_k,
        candidate_k=30,
        seed_papers=seed_papers,
        expand_k=expand_k,
        max_chunks_per_graph_paper=max_chunks_per_graph_paper,
        max_total_evidence_chunks=max_total_evidence_chunks,
        retrieval_weight=retrieval_weight,
        graph_weight=graph_weight,
        quality_weight=quality_weight,
        save_trace=save_trace
    )

    # Normalize graph expansion for clearer API trace
    expanded = trace.get("graph_trace", {}).get("expanded_papers", [])
    max_graph_raw = max([p.get("graph_score", 0.0) for p in expanded], default=1.0)

    normalized_expanded_papers = []

    for p in expanded:
        raw_score = p.get("graph_score", 0.0)

        normalized_expanded_papers.append({
            "paper_id": p.get("paper_id"),
            "graph_score_raw": raw_score,
            "graph_score_norm": round(raw_score / max_graph_raw, 4) if max_graph_raw else 0.0,
            "reasons": p.get("reasons", [])
        })

    response = {
        "api_version": "notebook_ask_v1",
        "run_id": trace.get("run_id"),
        "mode": trace.get("mode"),
        "question": trace.get("question"),

        "answer": trace.get("answer"),
        "citations": trace.get("citations", []),

        "parameters": trace.get("parameters", {}),
        "timing": trace.get("timing", {}),
        "quality_checks": trace.get("quality_checks", {}),

        "final_evidence": trace.get("final_evidence", []),

        "retrieval_trace": {
            "seed_chunks": trace.get("retrieval_trace", {}).get("seed_chunks", []),
            "seed_paper_ids": trace.get("retrieval_trace", {}).get("seed_paper_ids", [])
        } if return_trace else None,

        "graph_trace": {
            "enabled": trace.get("graph_trace", {}).get("enabled", False),
            "path_type": trace.get("graph_trace", {}).get("path_type"),
            "expanded_papers": normalized_expanded_papers
        } if return_trace else None,
    }

    if include_candidate_evidence:
        response["candidate_evidence"] = trace.get("candidate_evidence", [])

    if not return_trace:
        response.pop("retrieval_trace", None)
        response.pop("graph_trace", None)

    return response

In [21]:
question = "How do recent papers evaluate retrieval augmented generation?"

ask_result = ask(
    question=question,
    mode="hybrid_graphrag",
    return_trace=True,
    include_candidate_evidence=False
)

print("Mode:", ask_result["mode"])
print("Run ID:", ask_result["run_id"])
print("Answer:\n")
print(ask_result["answer"])

Mode: hybrid_graphrag
Run ID: 8fd4d7b5-be81-4da3-8828-aa8c90e3bca6
Answer:

Question: How do recent papers evaluate retrieval augmented generation?

Based on the selected evidence, the answer is supported by these papers:

- Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation pages 12: arXiv preprint arXiv:2310.13548 (2023). [63] Marco Siino, Mariana Falco, Daniele Croce, and Paolo Rosso. 2025. Exploring llms applications in law: A literature review on current legal nlp approaches. IEEE Access (2025). [64] Weihang Su, Yichen Tang, Qingyao Ai, Junxi Yan, Changyue Wang, Hongning Wan
- Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning pages 10: Kristina Toutanova. 2019. BERT: Pre-training of deep bidirectional transformers for language under- standing. In Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Tech- nologies, Volume 1 (

In [22]:
citations_df = pd.DataFrame(ask_result["citations"])

display(citations_df)

,rank,paper_id,title,chunk_id,pages,source,graph_reasons
0,1,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,paper85_chunk_0027,12,vector,[]
1,2,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,paper70_chunk_0018,10,graph,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."
2,3,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0009,6-7,vector,[]
3,4,paper63,FT-RAG: A Fine-grained Retrieval-Augmented Generation Framework for Complex Table Reasoning,paper63_chunk_0017,8-9,graph,"[shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_topic: Information Retrie..."
4,5,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0010,7,vector,[]
5,6,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0000,1-2,vector,[]
6,7,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0017,14,vector,[]
7,8,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,paper70_chunk_0000,1,graph,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."


In [23]:
final_evidence_df = pd.DataFrame(ask_result["final_evidence"])

display(
    final_evidence_df[
        [
            "rank",
            "source",
            "paper_id",
            "title",
            "page_start",
            "page_end",
            "retrieval_score",
            "graph_score",
            "quality_score",
            "final_score",
            "graph_reasons"
        ]
    ]
)

,rank,source,paper_id,title,page_start,page_end,retrieval_score,graph_score,quality_score,final_score,graph_reasons
0,1,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,12,12,1.000000,0.0,1.0,0.9000,[]
1,2,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,10,10,0.823266,1.0,1.0,0.8586,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."
2,3,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,6,7,0.926649,0.0,1.0,0.8413,[]
3,4,graph,paper63,FT-RAG: A Fine-grained Retrieval-Augmented Generation Framework for Complex Table Reasoning,8,9,0.777740,1.0,1.0,0.8222,"[shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_topic: Information Retrie..."
4,5,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,7,7,0.894741,0.0,1.0,0.8158,[]
5,6,vector,paper12,Generation: A Controlled Empirical Study1,1,2,0.891289,0.0,1.0,0.8130,[]
6,7,vector,paper12,Generation: A Controlled Empirical Study1,14,14,0.877060,0.0,1.0,0.8016,[]
7,8,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,1,1,0.739531,1.0,1.0,0.7916,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."


In [24]:
def validate_ask_response(response):
    required_top_keys = [
        "api_version",
        "run_id",
        "mode",
        "question",
        "answer",
        "citations",
        "parameters",
        "timing",
        "quality_checks",
        "final_evidence"
    ]

    required_citation_keys = [
        "rank",
        "paper_id",
        "title",
        "chunk_id",
        "pages",
        "source",
        "graph_reasons"
    ]

    required_evidence_keys = [
        "rank",
        "chunk_id",
        "paper_id",
        "title",
        "page_start",
        "page_end",
        "source",
        "retrieval_score",
        "graph_score",
        "quality_score",
        "final_score",
        "snippet"
    ]

    checks = []

    for key in required_top_keys:
        checks.append({
            "check": f"top_level_key:{key}",
            "passed": key in response
        })

    citations = response.get("citations", [])

    checks.append({
        "check": "has_citations",
        "passed": len(citations) > 0
    })

    for i, citation in enumerate(citations):
        for key in required_citation_keys:
            checks.append({
                "check": f"citation_{i+1}_has:{key}",
                "passed": key in citation
            })

    final_evidence = response.get("final_evidence", [])

    checks.append({
        "check": "has_final_evidence",
        "passed": len(final_evidence) > 0
    })

    for i, item in enumerate(final_evidence):
        for key in required_evidence_keys:
            checks.append({
                "check": f"evidence_{i+1}_has:{key}",
                "passed": key in item
            })

    checks.append({
        "check": "all_citations_have_page_ranges",
        "passed": all(c.get("pages") not in [None, "", "unknown"] for c in citations)
    })

    checks.append({
        "check": "citation_coverage_is_1",
        "passed": response.get("quality_checks", {}).get("citation_coverage") == 1.0
    })

    return pd.DataFrame(checks)


schema_check_df = validate_ask_response(ask_result)

display(schema_check_df)

print("Schema passed:", schema_check_df["passed"].all())

,check,passed
0,top_level_key:api_version,True
1,top_level_key:run_id,True
2,top_level_key:mode,True
3,top_level_key:question,True
4,top_level_key:answer,True
...,...,...
161,evidence_8_has:quality_score,True
162,evidence_8_has:final_score,True
163,evidence_8_has:snippet,True
164,all_citations_have_page_ranges,True


Schema passed: True


In [25]:
question = "How do recent papers evaluate retrieval augmented generation?"

ask_runs = {}

for mode in ["vector_only", "graph_guided", "hybrid_graphrag"]:
    ask_runs[mode] = ask(
        question=question,
        mode=mode,
        top_k=5,
        seed_papers=3,
        expand_k=5,
        max_total_evidence_chunks=8,
        return_trace=True,
        include_candidate_evidence=False,
        save_trace=True
    )

print("Completed modes:", list(ask_runs.keys()))

Completed modes: ['vector_only', 'graph_guided', 'hybrid_graphrag']


In [26]:
def summarize_ask_response(response):
    final = response.get("final_evidence", [])
    sources = [x.get("source") for x in final]
    params = response.get("parameters", {})
    timing = response.get("timing", {})
    graph_trace = response.get("graph_trace", {}) or {}

    return {
        "mode": response.get("mode"),
        "retrieval_weight": params.get("retrieval_weight"),
        "graph_weight": params.get("graph_weight"),
        "quality_weight": params.get("quality_weight"),
        "use_graph": params.get("use_graph"),

        "num_final_chunks": len(final),
        "vector_chunks": sources.count("vector"),
        "graph_chunks": sources.count("graph"),
        "both_chunks": sources.count("both"),
        "unique_papers": len(set(x.get("paper_id") for x in final)),

        "num_graph_expanded_papers": len(graph_trace.get("expanded_papers", [])),
        "citation_coverage": response.get("quality_checks", {}).get("citation_coverage"),
        "has_page_ranges": response.get("quality_checks", {}).get("has_page_ranges"),

        "retrieval_ms": timing.get("retrieval_ms"),
        "graph_ms": timing.get("graph_ms"),
        "total_ms": timing.get("total_ms"),
    }


mode_comparison_df = pd.DataFrame([
    summarize_ask_response(ask_runs["vector_only"]),
    summarize_ask_response(ask_runs["graph_guided"]),
    summarize_ask_response(ask_runs["hybrid_graphrag"]),
])

display(mode_comparison_df)

mode_comparison_path = EXECUTOR_CACHE / "ask_mode_comparison_table.csv"
mode_comparison_df.to_csv(mode_comparison_path, index=False)

print("Saved:", mode_comparison_path)

,mode,retrieval_weight,graph_weight,quality_weight,use_graph,num_final_chunks,vector_chunks,graph_chunks,both_chunks,unique_papers,num_graph_expanded_papers,citation_coverage,has_page_ranges,retrieval_ms,graph_ms,total_ms
0,vector_only,1.0,0.0,0.0,False,5,5,0,0,3,0,1.0,True,86.88,0.00,86.88
1,graph_guided,0.5,0.4,0.1,True,8,1,7,0,5,5,1.0,True,71.86,87.01,158.87
2,hybrid_graphrag,0.8,0.1,0.1,True,8,5,3,0,5,5,1.0,True,58.73,79.51,138.24


Saved: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\ask_mode_comparison_table.csv


In [27]:
interpretation_df = pd.DataFrame([
    {
        "mode": "vector_only",
        "interpretation": (
            "Baseline mode. It uses only the retrieved chunks and disables graph expansion. "
            "This is useful for checking whether graph expansion adds useful context."
        )
    },
    {
        "mode": "graph_guided",
        "interpretation": (
            "Graph-heavy ablation. It shows what happens when graph-expanded evidence has strong influence. "
            "Useful for comparison, but not selected as the safest default because graph chunks can dominate."
        )
    },
    {
        "mode": "hybrid_graphrag",
        "interpretation": (
            "Balanced default. Retrieval remains the main relevance signal while graph expansion adds supporting "
            "papers through explainable shared-topic paths. This should produce mixed evidence rather than graph-only evidence."
        )
    }
])

display(interpretation_df)

,mode,interpretation
0,vector_only,Baseline mode. It uses only the retrieved chunks and disables graph expansion. This is useful for checking whether g...
1,graph_guided,Graph-heavy ablation. It shows what happens when graph-expanded evidence has strong influence. Useful for comparison...
2,hybrid_graphrag,Balanced default. Retrieval remains the main relevance signal while graph expansion adds supporting papers through e...


In [28]:
sample_response_path = EXECUTOR_CACHE / "ask_sample_hybrid_graphrag.json"

with open(sample_response_path, "w", encoding="utf-8") as f:
    json.dump(ask_runs["hybrid_graphrag"], f, indent=2, ensure_ascii=False)

print("Saved sample /ask response:", sample_response_path)

Saved sample /ask response: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\ask_sample_hybrid_graphrag.json


In [29]:
TEST_QUESTIONS = [
    {
        "question_id": "q_api_001",
        "question": "How do recent papers evaluate retrieval augmented generation?",
        "reason": "Tests RAG evaluation and retrieval-related graph expansion."
    },
    {
        "question_id": "q_api_002",
        "question": "What methods are used for reranking in retrieval augmented generation?",
        "reason": "Tests whether graph expansion finds reranking-related papers."
    },
    {
        "question_id": "q_api_003",
        "question": "How do recent papers discuss hallucination detection and safety in large language models?",
        "reason": "Tests whether graph expansion adds safety and hallucination-related context."
    }
]

pd.DataFrame(TEST_QUESTIONS)

,question_id,question,reason
0,q_api_001,How do recent papers evaluate retrieval augmented generation?,Tests RAG evaluation and retrieval-related graph expansion.
1,q_api_002,What methods are used for reranking in retrieval augmented generation?,Tests whether graph expansion finds reranking-related papers.
2,q_api_003,How do recent papers discuss hallucination detection and safety in large language models?,Tests whether graph expansion adds safety and hallucination-related context.


In [30]:
API_MODES = [
    "vector_only",
    "graph_guided",
    "hybrid_graphrag"
]

api_runs = []

for q in TEST_QUESTIONS:
    for mode in API_MODES:
        print(f"Running {q['question_id']} | {mode}")

        result = ask(
            question=q["question"],
            mode=mode,
            top_k=5,
            seed_papers=3,
            expand_k=5,
            max_chunks_per_graph_paper=2,
            max_total_evidence_chunks=8,
            return_trace=True,
            include_candidate_evidence=False,
            save_trace=True
        )

        result["question_id"] = q["question_id"]
        result["question_reason"] = q["reason"]

        api_runs.append(result)

print("Total API-style runs:", len(api_runs))

Running q_api_001 | vector_only
Running q_api_001 | graph_guided
Running q_api_001 | hybrid_graphrag
Running q_api_002 | vector_only
Running q_api_002 | graph_guided
Running q_api_002 | hybrid_graphrag
Running q_api_003 | vector_only
Running q_api_003 | graph_guided
Running q_api_003 | hybrid_graphrag
Total API-style runs: 9


In [31]:
validation_rows = []

for run in api_runs:
    schema_df = validate_ask_response(run)

    validation_rows.append({
        "question_id": run["question_id"],
        "mode": run["mode"],
        "schema_passed": schema_df["passed"].all(),
        "failed_checks": schema_df[schema_df["passed"] == False]["check"].tolist()
    })

validation_df = pd.DataFrame(validation_rows)

display(validation_df)

print("All API outputs valid:", validation_df["schema_passed"].all())

,question_id,mode,schema_passed,failed_checks
0,q_api_001,vector_only,True,[]
1,q_api_001,graph_guided,True,[]
2,q_api_001,hybrid_graphrag,True,[]
3,q_api_002,vector_only,True,[]
4,q_api_002,graph_guided,True,[]
5,q_api_002,hybrid_graphrag,True,[]
6,q_api_003,vector_only,True,[]
7,q_api_003,graph_guided,True,[]
8,q_api_003,hybrid_graphrag,True,[]


All API outputs valid: True


In [32]:
def summarize_api_run(run):
    final = run.get("final_evidence", [])
    sources = [x.get("source") for x in final]

    params = run.get("parameters", {})
    timing = run.get("timing", {})
    quality = run.get("quality_checks", {})
    graph_trace = run.get("graph_trace") or {}

    return {
        "question_id": run.get("question_id"),
        "question": run.get("question"),
        "mode": run.get("mode"),

        "retrieval_weight": params.get("retrieval_weight"),
        "graph_weight": params.get("graph_weight"),
        "quality_weight": params.get("quality_weight"),
        "use_graph": params.get("use_graph"),

        "num_final_chunks": len(final),
        "vector_chunks": sources.count("vector"),
        "graph_chunks": sources.count("graph"),
        "both_chunks": sources.count("both"),
        "unique_papers": len(set(x.get("paper_id") for x in final)),

        "num_graph_expanded_papers": len(graph_trace.get("expanded_papers", [])),
        "citation_coverage": quality.get("citation_coverage"),
        "has_page_ranges": quality.get("has_page_ranges"),

        "retrieval_ms": timing.get("retrieval_ms"),
        "graph_ms": timing.get("graph_ms"),
        "total_ms": timing.get("total_ms"),

        "top_1_source": final[0].get("source") if final else None,
        "top_1_paper": final[0].get("paper_id") if final else None,
        "top_1_title": final[0].get("title") if final else None,
    }


api_comparison_df = pd.DataFrame([
    summarize_api_run(run)
    for run in api_runs
])

display(api_comparison_df)

api_comparison_path = EXECUTOR_CACHE / "api_ready_3_question_mode_comparison.csv"
api_comparison_df.to_csv(api_comparison_path, index=False)

print("Saved:", api_comparison_path)

,question_id,question,mode,retrieval_weight,graph_weight,quality_weight,use_graph,num_final_chunks,vector_chunks,graph_chunks,...,unique_papers,num_graph_expanded_papers,citation_coverage,has_page_ranges,retrieval_ms,graph_ms,total_ms,top_1_source,top_1_paper,top_1_title
0,q_api_001,How do recent papers evaluate retrieval augmented generation?,vector_only,1.0,0.0,0.0,False,5,5,0,...,3,0,1.0,True,270.58,0.00,270.58,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation
1,q_api_001,How do recent papers evaluate retrieval augmented generation?,graph_guided,0.5,0.4,0.1,True,8,1,7,...,5,5,1.0,True,160.11,105.01,290.64,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning
2,q_api_001,How do recent papers evaluate retrieval augmented generation?,hybrid_graphrag,0.8,0.1,0.1,True,8,5,3,...,5,5,1.0,True,110.51,122.22,232.73,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation
3,q_api_002,What methods are used for reranking in retrieval augmented generation?,vector_only,1.0,0.0,0.0,False,5,5,0,...,3,0,1.0,True,151.20,0.00,151.20,vector,paper12,Generation: A Controlled Empirical Study1
4,q_api_002,What methods are used for reranking in retrieval augmented generation?,graph_guided,0.5,0.4,0.1,True,8,2,6,...,5,5,1.0,True,72.33,112.51,184.84,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning
5,q_api_002,What methods are used for reranking in retrieval augmented generation?,hybrid_graphrag,0.8,0.1,0.1,True,8,5,3,...,5,5,1.0,True,66.87,89.38,156.24,vector,paper12,Generation: A Controlled Empirical Study1
6,q_api_003,How do recent papers discuss hallucination detection and safety in large language models?,vector_only,1.0,0.0,0.0,False,5,5,0,...,3,0,1.0,True,111.60,0.00,111.60,vector,paper19,HalluScan: A Systematic Benchmark for Detecting and Mitigating Hallucinations in Instruction-Following LLMs
7,q_api_003,How do recent papers discuss hallucination detection and safety in large language models?,graph_guided,0.5,0.4,0.1,True,8,2,6,...,5,5,1.0,True,83.50,95.51,179.01,graph,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models
8,q_api_003,How do recent papers discuss hallucination detection and safety in large language models?,hybrid_graphrag,0.8,0.1,0.1,True,8,5,3,...,5,5,1.0,True,71.93,83.36,155.28,vector,paper19,HalluScan: A Systematic Benchmark for Detecting and Mitigating Hallucinations in Instruction-Following LLMs


Saved: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\api_ready_3_question_mode_comparison.csv


In [33]:
def compact_final_evidence(run):
    rows = []

    for item in run.get("final_evidence", []):
        rows.append({
            "question_id": run.get("question_id"),
            "mode": run.get("mode"),
            "rank": item.get("rank"),
            "source": item.get("source"),
            "paper_id": item.get("paper_id"),
            "title": item.get("title"),
            "pages": (
                str(item.get("page_start"))
                if item.get("page_start") == item.get("page_end")
                else f"{item.get('page_start')}-{item.get('page_end')}"
            ),
            "retrieval_score": item.get("retrieval_score"),
            "graph_score": item.get("graph_score"),
            "final_score": item.get("final_score"),
            "graph_reasons": item.get("graph_reasons"),
        })

    return rows


final_evidence_all_df = pd.DataFrame([
    row
    for run in api_runs
    for row in compact_final_evidence(run)
])

display(final_evidence_all_df)

final_evidence_path = EXECUTOR_CACHE / "api_ready_3_question_final_evidence.csv"
final_evidence_all_df.to_csv(final_evidence_path, index=False)

print("Saved:", final_evidence_path)

,question_id,mode,rank,source,paper_id,title,pages,retrieval_score,graph_score,final_score,graph_reasons
0,q_api_001,vector_only,1,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,12,1.000000,0.00,1.0000,[]
1,q_api_001,vector_only,2,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,6-7,0.403361,0.00,0.4034,[]
2,q_api_001,vector_only,3,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,7,0.143817,0.00,0.1438,[]
3,q_api_001,vector_only,4,vector,paper12,Generation: A Controlled Empirical Study1,1-2,0.115743,0.00,0.1157,[]
4,q_api_001,vector_only,5,vector,paper12,Generation: A Controlled Empirical Study1,14,0.000000,0.00,0.0000,[]
...,...,...,...,...,...,...,...,...,...,...,...
58,q_api_003,hybrid_graphrag,4,vector,paper19,HalluScan: A Systematic Benchmark for Detecting and Mitigating Hallucinations in Instruction-Following LLMs,7-8,0.799786,0.00,0.7398,[]
59,q_api_003,hybrid_graphrag,5,vector,paper67,Hallucinations Undermine Trust; Metacognition is a Way Forward,16,0.793324,0.00,0.7347,[]
60,q_api_003,hybrid_graphrag,6,graph,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,10-11,0.489932,0.75,0.5669,"[shared_topic: Large Language Models, shared_topic: LLM Evaluation, shared_topic: Multilingual NLP]"
61,q_api_003,hybrid_graphrag,7,graph,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,1,0.414186,0.75,0.5063,"[shared_topic: Large Language Models, shared_topic: LLM Evaluation, shared_topic: Multilingual NLP]"


Saved: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\api_ready_3_question_final_evidence.csv


In [34]:
def compact_citations(run):
    rows = []

    for c in run.get("citations", []):
        rows.append({
            "question_id": run.get("question_id"),
            "mode": run.get("mode"),
            "rank": c.get("rank"),
            "paper_id": c.get("paper_id"),
            "title": c.get("title"),
            "chunk_id": c.get("chunk_id"),
            "pages": c.get("pages"),
            "source": c.get("source"),
            "graph_reasons": c.get("graph_reasons"),
        })

    return rows


citations_all_df = pd.DataFrame([
    row
    for run in api_runs
    for row in compact_citations(run)
])

display(citations_all_df)

citations_path = EXECUTOR_CACHE / "api_ready_3_question_citations.csv"
citations_all_df.to_csv(citations_path, index=False)

print("Saved:", citations_path)

,question_id,mode,rank,paper_id,title,chunk_id,pages,source,graph_reasons
0,q_api_001,vector_only,1,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,paper85_chunk_0027,12,vector,[]
1,q_api_001,vector_only,2,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0009,6-7,vector,[]
2,q_api_001,vector_only,3,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0010,7,vector,[]
3,q_api_001,vector_only,4,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0000,1-2,vector,[]
4,q_api_001,vector_only,5,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0017,14,vector,[]
...,...,...,...,...,...,...,...,...,...
58,q_api_003,hybrid_graphrag,4,paper19,HalluScan: A Systematic Benchmark for Detecting and Mitigating Hallucinations in Instruction-Following LLMs,paper19_chunk_0007,7-8,vector,[]
59,q_api_003,hybrid_graphrag,5,paper67,Hallucinations Undermine Trust; Metacognition is a Way Forward,paper67_chunk_0021,16,vector,[]
60,q_api_003,hybrid_graphrag,6,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,paper146_chunk_0016,10-11,graph,"[shared_topic: Large Language Models, shared_topic: LLM Evaluation, shared_topic: Multilingual NLP]"
61,q_api_003,hybrid_graphrag,7,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,paper146_chunk_0000,1,graph,"[shared_topic: Large Language Models, shared_topic: LLM Evaluation, shared_topic: Multilingual NLP]"


Saved: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\api_ready_3_question_citations.csv


In [35]:
api_response_dir = EXECUTOR_CACHE / "api_ready_3_question_responses"
api_response_dir.mkdir(parents=True, exist_ok=True)

for run in api_runs:
    out_name = f"{run['question_id']}_{run['mode']}.json"
    out_path = api_response_dir / out_name

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2, ensure_ascii=False)

print("Saved API-ready JSON responses to:", api_response_dir)

Saved API-ready JSON responses to: D:\Year4\Term3\topicsOfAI\project\CLPapersAIAgent\notebook_cache\05_graphrag_executor\api_ready_3_question_responses


# Final Proof: API-ready GraphRAG Executor Results

This section loads the saved API-ready results without rerunning the executor.  
It is used as proof for the instructor.

In [36]:
from pathlib import Path
import json
import pandas as pd

EXECUTOR_CACHE = Path("../notebook_cache/05_graphrag_executor")

proof_files = {
    "mode_comparison": EXECUTOR_CACHE / "api_ready_3_question_mode_comparison.csv",
    "final_evidence": EXECUTOR_CACHE / "api_ready_3_question_final_evidence.csv",
    "citations": EXECUTOR_CACHE / "api_ready_3_question_citations.csv",
    "api_response_folder": EXECUTOR_CACHE / "api_ready_3_question_responses",
    "sample_hybrid_response": EXECUTOR_CACHE / "ask_sample_hybrid_graphrag.json",
}

proof_check_df = pd.DataFrame([
    {
        "proof_item": name,
        "path": str(path),
        "exists": path.exists()
    }
    for name, path in proof_files.items()
])

display(proof_check_df)

if not proof_check_df["exists"].all():
    raise FileNotFoundError("Some final proof files are missing.")
else:
    print("All final proof files found. No rerun is needed.")

,proof_item,path,exists
0,mode_comparison,..\notebook_cache\05_graphrag_executor\api_ready_3_question_mode_comparison.csv,True
1,final_evidence,..\notebook_cache\05_graphrag_executor\api_ready_3_question_final_evidence.csv,True
2,citations,..\notebook_cache\05_graphrag_executor\api_ready_3_question_citations.csv,True
3,api_response_folder,..\notebook_cache\05_graphrag_executor\api_ready_3_question_responses,True
4,sample_hybrid_response,..\notebook_cache\05_graphrag_executor\ask_sample_hybrid_graphrag.json,True


All final proof files found. No rerun is needed.


In [37]:
mode_comparison_df = pd.read_csv(proof_files["mode_comparison"])

mode_display_cols = [
    "question_id",
    "mode",
    "use_graph",
    "retrieval_weight",
    "graph_weight",
    "quality_weight",
    "num_final_chunks",
    "vector_chunks",
    "graph_chunks",
    "unique_papers",
    "citation_coverage",
    "has_page_ranges",
    "total_ms",
    "top_1_source",
    "top_1_paper",
]

display(mode_comparison_df[mode_display_cols])

,question_id,mode,use_graph,retrieval_weight,graph_weight,quality_weight,num_final_chunks,vector_chunks,graph_chunks,unique_papers,citation_coverage,has_page_ranges,total_ms,top_1_source,top_1_paper
0,q_api_001,vector_only,False,1.0,0.0,0.0,5,5,0,3,1.0,True,270.58,vector,paper85
1,q_api_001,graph_guided,True,0.5,0.4,0.1,8,1,7,5,1.0,True,290.64,graph,paper70
2,q_api_001,hybrid_graphrag,True,0.8,0.1,0.1,8,5,3,5,1.0,True,232.73,vector,paper85
3,q_api_002,vector_only,False,1.0,0.0,0.0,5,5,0,3,1.0,True,151.20,vector,paper12
4,q_api_002,graph_guided,True,0.5,0.4,0.1,8,2,6,5,1.0,True,184.84,graph,paper70
5,q_api_002,hybrid_graphrag,True,0.8,0.1,0.1,8,5,3,5,1.0,True,156.24,vector,paper12
6,q_api_003,vector_only,False,1.0,0.0,0.0,5,5,0,3,1.0,True,111.60,vector,paper19
7,q_api_003,graph_guided,True,0.5,0.4,0.1,8,2,6,5,1.0,True,179.01,graph,paper146
8,q_api_003,hybrid_graphrag,True,0.8,0.1,0.1,8,5,3,5,1.0,True,155.28,vector,paper19


In [38]:
final_evidence_df = pd.read_csv(proof_files["final_evidence"])

final_evidence_display_cols = [
    "question_id",
    "mode",
    "rank",
    "source",
    "paper_id",
    "title",
    "pages",
    "retrieval_score",
    "graph_score",
    "final_score",
    "graph_reasons",
]

display(final_evidence_df[final_evidence_display_cols])

,question_id,mode,rank,source,paper_id,title,pages,retrieval_score,graph_score,final_score,graph_reasons
0,q_api_001,vector_only,1,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,12,1.000000,0.00,1.0000,[]
1,q_api_001,vector_only,2,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,6-7,0.403361,0.00,0.4034,[]
2,q_api_001,vector_only,3,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,7,0.143817,0.00,0.1438,[]
3,q_api_001,vector_only,4,vector,paper12,Generation: A Controlled Empirical Study1,1-2,0.115743,0.00,0.1157,[]
4,q_api_001,vector_only,5,vector,paper12,Generation: A Controlled Empirical Study1,14,0.000000,0.00,0.0000,[]
...,...,...,...,...,...,...,...,...,...,...,...
58,q_api_003,hybrid_graphrag,4,vector,paper19,HalluScan: A Systematic Benchmark for Detecting and Mitigating Hallucinations in Instruction-Following LLMs,7-8,0.799786,0.00,0.7398,[]
59,q_api_003,hybrid_graphrag,5,vector,paper67,Hallucinations Undermine Trust; Metacognition is a Way Forward,16,0.793324,0.00,0.7347,[]
60,q_api_003,hybrid_graphrag,6,graph,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,10-11,0.489932,0.75,0.5669,"['shared_topic: Large Language Models', 'shared_topic: LLM Evaluation', 'shared_topic: Multilingual NLP']"
61,q_api_003,hybrid_graphrag,7,graph,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,1,0.414186,0.75,0.5063,"['shared_topic: Large Language Models', 'shared_topic: LLM Evaluation', 'shared_topic: Multilingual NLP']"


In [39]:
citations_df = pd.read_csv(proof_files["citations"])

citation_display_cols = [
    "question_id",
    "mode",
    "rank",
    "paper_id",
    "title",
    "chunk_id",
    "pages",
    "source",
    "graph_reasons",
]

display(citations_df[citation_display_cols])

,question_id,mode,rank,paper_id,title,chunk_id,pages,source,graph_reasons
0,q_api_001,vector_only,1,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,paper85_chunk_0027,12,vector,[]
1,q_api_001,vector_only,2,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0009,6-7,vector,[]
2,q_api_001,vector_only,3,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0010,7,vector,[]
3,q_api_001,vector_only,4,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0000,1-2,vector,[]
4,q_api_001,vector_only,5,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0017,14,vector,[]
...,...,...,...,...,...,...,...,...,...
58,q_api_003,hybrid_graphrag,4,paper19,HalluScan: A Systematic Benchmark for Detecting and Mitigating Hallucinations in Instruction-Following LLMs,paper19_chunk_0007,7-8,vector,[]
59,q_api_003,hybrid_graphrag,5,paper67,Hallucinations Undermine Trust; Metacognition is a Way Forward,paper67_chunk_0021,16,vector,[]
60,q_api_003,hybrid_graphrag,6,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,paper146_chunk_0016,10-11,graph,"['shared_topic: Large Language Models', 'shared_topic: LLM Evaluation', 'shared_topic: Multilingual NLP']"
61,q_api_003,hybrid_graphrag,7,paper146,ML-Bench&Guard: Policy-Grounded Multilingual Safety Benchmark and Guardrail for Large Language Models,paper146_chunk_0000,1,graph,"['shared_topic: Large Language Models', 'shared_topic: LLM Evaluation', 'shared_topic: Multilingual NLP']"


In [40]:
sample_path = proof_files["sample_hybrid_response"]

with open(sample_path, "r", encoding="utf-8") as f:
    sample_ask_response = json.load(f)

print("Loaded sample response:", sample_path)
print("Mode:", sample_ask_response["mode"])
print("Question:", sample_ask_response["question"])

print("\nTop-level API keys:")
for key in sample_ask_response.keys():
    print("-", key)

print("\nQuality checks:")
print(json.dumps(sample_ask_response["quality_checks"], indent=2))

Loaded sample response: ..\notebook_cache\05_graphrag_executor\ask_sample_hybrid_graphrag.json
Mode: hybrid_graphrag
Question: How do recent papers evaluate retrieval augmented generation?

Top-level API keys:
- api_version
- run_id
- mode
- question
- answer
- citations
- parameters
- timing
- quality_checks
- final_evidence
- retrieval_trace
- graph_trace

Quality checks:
{
  "num_seed_chunks": 5,
  "num_seed_papers": 3,
  "num_graph_papers": 5,
  "num_candidate_chunks": 15,
  "num_final_chunks": 8,
  "citation_coverage": 1.0,
  "has_page_ranges": true
}


In [41]:
print("Sample API-ready answer:")
print(sample_ask_response["answer"])

print("\nSample citations:")
display(pd.DataFrame(sample_ask_response["citations"]))

print("\nSample final evidence:")
sample_final_evidence_df = pd.DataFrame(sample_ask_response["final_evidence"])

display(
    sample_final_evidence_df[
        [
            "rank",
            "source",
            "paper_id",
            "title",
            "page_start",
            "page_end",
            "retrieval_score",
            "graph_score",
            "final_score",
            "graph_reasons",
        ]
    ]
)

Sample API-ready answer:
Question: How do recent papers evaluate retrieval augmented generation?

Based on the selected evidence, the answer is supported by these papers:

- Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation pages 12: arXiv preprint arXiv:2310.13548 (2023). [63] Marco Siino, Mariana Falco, Daniele Croce, and Paolo Rosso. 2025. Exploring llms applications in law: A literature review on current legal nlp approaches. IEEE Access (2025). [64] Weihang Su, Yichen Tang, Qingyao Ai, Junxi Yan, Changyue Wang, Hongning Wan
- Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning pages 10: Kristina Toutanova. 2019. BERT: Pre-training of deep bidirectional transformers for language under- standing. In Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Tech- nologies, Volume 1 (Long and Short Papers), pages 4
- H-RAG at SemEval-

,rank,paper_id,title,chunk_id,pages,source,graph_reasons
0,1,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,paper85_chunk_0027,12,vector,[]
1,2,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,paper70_chunk_0018,10,graph,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."
2,3,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0009,6-7,vector,[]
3,4,paper63,FT-RAG: A Fine-grained Retrieval-Augmented Generation Framework for Complex Table Reasoning,paper63_chunk_0017,8-9,graph,"[shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_topic: Information Retrie..."
4,5,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,paper148_chunk_0010,7,vector,[]
5,6,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0000,1-2,vector,[]
6,7,paper12,Generation: A Controlled Empirical Study1,paper12_chunk_0017,14,vector,[]
7,8,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,paper70_chunk_0000,1,graph,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."



Sample final evidence:


,rank,source,paper_id,title,page_start,page_end,retrieval_score,graph_score,final_score,graph_reasons
0,1,vector,paper85,Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation,12,12,1.000000,0.0,0.9000,[]
1,2,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,10,10,0.823266,1.0,0.8586,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."
2,3,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,6,7,0.926649,0.0,0.8413,[]
3,4,graph,paper63,FT-RAG: A Fine-grained Retrieval-Augmented Generation Framework for Complex Table Reasoning,8,9,0.777740,1.0,0.8222,"[shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_topic: Information Retrie..."
4,5,vector,paper148,H-RAG at SemEval-2026 Task 8: Hierarchical Parent-Child Retrieval for Multi-Turn RAG Conversations,7,7,0.894741,0.0,0.8158,[]
5,6,vector,paper12,Generation: A Controlled Empirical Study1,1,2,0.891289,0.0,0.8130,[]
6,7,vector,paper12,Generation: A Controlled Empirical Study1,14,14,0.877060,0.0,0.8016,[]
7,8,graph,paper70,Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning,1,1,0.739531,1.0,0.7916,"[shared_topic: Reranking, shared_topic: Retrieval-Augmented Generation, shared_topic: Large Language Models, shared_..."


### Final Interpretation

The executor was tested using a notebook-level API-ready `/ask` flow.

The results were generated for three questions and three modes: vector-only, graph-guided, and hybrid GraphRAG.

Vector-only is the baseline because graph expansion is disabled.

Graph-guided is the graph-heavy ablation because graph evidence has stronger influence.

Hybrid GraphRAG is the default setting because retrieval remains the main relevance signal.

The mode comparison table shows the difference between retrieval-only, graph-guided, and balanced hybrid behavior.

The final evidence table shows the selected chunks, scores, sources, page ranges, and graph reasons.

The citation table confirms that each answer can be traced to a paper, chunk, and page range.

The saved `/ask` sample response proves that the executor output is ready to be moved into FastAPI later.
